In [103]:
import json
from pathlib import Path
import pandas as pd
import re
import extract as ext
from shutil import move

In [104]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [105]:
form_classified = pd.read_csv("./csv/form_classified.csv", encoding="utf-8")
form_emestrada = pd.read_csv("./csv/form_emestrada.csv", encoding="utf-8")

In [106]:
form_emestrada.head()

,year,exam_name,statement
0,2010,química - 2010 - junio,a) Nitrito de plata b) Hidróxido de magnesio c...
1,2010,química - 2010 - junio,a) Peróxido de rubidio b) Hidrogenocarbonato d...
2,2010,química - 2010 - reserva 1,a) Permanganato de bario b) Peróxido de potasi...
3,2010,química - 2010 - reserva 1,a) Hidrogenocarbonato de sodio b) Hidróxido de...
4,2010,química - 2010 - reserva 2,a) Pentafluoruro de antimonio b) Óxido de plom...


In [107]:
form_classified.sample(5).statement

81     1.- Formule o nombre los siguientes compuestos...
166    A1. Formule o nombre los siguientes compuestos...
18     1.- Formule o nombre los compuestos siguientes...
153    A2. Formule o nombre los siguientes compuestos...
53     1.- Formule o nombre los siguientes compuestos...
Name: statement, dtype: str

### Formulación 2025

In [108]:
for i in form_classified.loc[form_classified.year == 2025, "statement"]:
    print(i)

4A. Nombre o formule los siguientes compuestos:
a) CH₂=CH―CH₂OH b) CH₃―C(CH₃)₂―C == C H c) Propanamida d) Ácido benzoico
4A. Nombre o formule los siguientes compuestos:
O
―
a) CH₃―CH₂―CH₂―CH₂―C―NH₂ b) C H₃― ―OH c) Butanodial d) 6-Bromohex-2-ino
4A. Nombre o formule los siguientes compuestos:
a) CH₂=CH―CH₂―NH₂ b) CH₃―O―CH₃ c) Etilbenceno d) Difenil éter
4A. Nombre o formule los siguientes compuestos:
a) CH₃―C=CH―CH₃ b) CH₃―CH₂―CH(OH)―CH₃ c) Etanoato de etilo d) Metil propil éter
―
CH₃
4A. Nombre o formule los siguientes compuestos:
a) CH₃―CH=CH―CH=CH₂ b) CH₃―CH(OH)―CH₂―CN c) 3–Metilpent–2–eno d) Etanamida
4A. Nombre o formule los siguientes compuestos:
a) CH₃―CH₂―O―CH₃ b) CH₃―CH―CH₂―CHO c) Etanamina d) Penta–1,3–dieno
―
CH₃


### Extraer nombramiento fórmulas orgánicas 2025

In [109]:
def extraer_nombrar_compuestos_25(statement):
    if not pd.isna(statement):
        start_position = statement.find("c)")
        end_position = statement.find("d) ")
        statement = statement[start_position:end_position]
        statement = statement.replace(";", "").replace("\n", " ").strip()
    
    return statement

In [110]:
form_classified[form_classified.year == 2025].statement.apply(extraer_nombrar_compuestos_25)

170          c) Propanamida
171           c) Butanodial
172          c) Etilbenceno
173    c) Etanoato de etilo
174    c) 3–Metilpent–2–eno
175            c) Etanamina
Name: statement, dtype: str

In [111]:
form_emestrada[form_emestrada.year == 2025].statement.apply(extraer_nombrar_compuestos_25)

178    c) 3-Metilpent-2-eno
179          c) Propanamida
180          c) Etilbenceno
181            c) Etanamina
Name: statement, dtype: str

In [112]:
mask = form_classified.year == 2025
form_classified.loc[mask, "statement"] = form_classified.loc[mask].statement.apply(extraer_nombrar_compuestos_25)

In [113]:
mask = form_emestrada.year == 2025
form_emestrada.loc[mask, "statement"] = form_emestrada.loc[mask].statement.apply(extraer_nombrar_compuestos_25)

## Extraer apartados de nombrar en ejercicios de fórmulación (**previos a 2025**)

In [114]:
def extraer_nombrar_compuestos(statement):
    if not pd.isna(statement):
        start_position = statement.find("a)")
        if " d) " in statement:
            end_position = statement.find(" d) ")
        else:
            end_position = -1
        statement = statement[start_position:end_position]
        # unir valencia con Stock
        statement = re.sub(r"\s+(\([IVX]+\))", r"\1", statement)
        statement = statement.replace(";", "").replace("\n", " ").strip()
        #statement = re.sub(r"[^0-9a-záéíóú]", "", statement.lower())
    
    return statement

In [115]:
for i in form_classified.statement.sample(5).apply(extraer_nombrar_compuestos):
    print(i)

a) Peróxido de hidrógeno b) Hidrogenosulfito de sodio c) Hidruro de boro
a) Ácido sulfuroso b) Hidróxido de cobre(II) c) 3,3-Dimetilciclopenteno
a) Dihidruro de cobalto b) Hipoyodito de mercurio(II) c) Ácido butanodioico
a) Dihidruro de plomo b) Ácido cloroso c) Pentano-2,4-diona d) Ag₂O e) K₂Cr₂O₇ f) CH₂OHCH₂OH
a) Cloruro de amonio b) Carbonato de rubidio c) Ciclopentano


In [116]:
for i in form_emestrada.statement.sample(5):
    print(i)

a) Óxido de manganeso(VII) b) Ácido peryódico c) 1,1,2-Trimetilciclopentano
a) Pentafluoruro de antimonio b) Fosfato de calcio c) Ácido butanodioico
a) Peróxido de bario b) Hidróxido de magnesio c) Etanamida
a) Hidróxido de estroncio b) Dicromato de bario c) Ácido 2,3-dihidroxibutanoico
a) Bromato de potasio b) Hidróxido de aluminio c) 4-Metilfenol


In [117]:
mask = form_classified.year != 2025
form_classified.loc[mask, "statement"] = form_classified.loc[mask].statement.apply(extraer_nombrar_compuestos)

## Combinar los datasets
Hacer *match* entre los ejercicios de formulación de *emestrada* y los clasificados, usando como clave el enunciado de formulación. 

In [118]:
# x: classified, y: emestrada
form_merged = pd.merge(form_classified, form_emestrada, on=["statement", "year"])

In [119]:
form_merged.shape

(122, 9)

In [120]:
form_merged.columns

Index(['year', 'exam_name_x', 'exercise_number', 'topic', 'exercise_type',
       'statement', 'exam_option', 'exam_file_path', 'exam_name_y'],
      dtype='str')

In [121]:
form_merged.year.aggregate(["min", "max"])

min    2010
max    2025
Name: year, dtype: int64

In [122]:
cols_to_keep = ["year", "exam_file_path", "exam_name_y"]
form_merged = form_merged.drop([col for col in form_merged.columns if col not in cols_to_keep], axis = 1)

In [123]:
form_merged = form_merged.rename(columns = {"exam_name_y" : "filename_emestrada",
                     "exam_file_path" : "source_file_path",})

In [124]:
form_merged.head()

,year,source_file_path,filename_emestrada
0,2010,examenes_quimica/2010_quimica/2010_quimica_qui...,química - 2010 - junio
1,2010,examenes_quimica/2010_quimica/2010_quimica_qui...,química - 2010 - reserva 1
2,2010,examenes_quimica/2010_quimica/2010_quimica_qui...,química - 2010 - reserva 1
3,2010,examenes_quimica/2010_quimica/2010_quimica_qui...,química - 2010 - reserva 2
4,2010,examenes_quimica/2010_quimica/2010_quimica_qui...,química - 2010 - septiembre


In [125]:
form_merged = form_merged.drop_duplicates()

## Renombrar archivos

In [126]:
for idx, row in form_merged.iterrows():
    old = ROOT / row.source_file_path
    new = old.parent / f"{row.filename_emestrada}.pdf"

    #print(f"{old} --> {new}")
    try:
        move(old, new)
    except Exception as e:
        print(f"Archivo: {old.name}. Error: {e}")